In [14]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import HistGradientBoostingClassifier
import json
import joblib
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier

In [15]:
train = pd.read_csv("train_cleaned.csv")
test = pd.read_csv("test_cleaned.csv")

In [16]:
# feature engineering

# train dataset
train['screen_time_gap'] = train['daily_screen_time_hours'] - train['sleep_hours']
train['weekend_screen_ratio'] = train['weekend_screen_time'] / (train['daily_screen_time_hours'] + 1e-5)
train['social_media_ratio'] = train['social_media_hours'] / (train['daily_screen_time_hours'] + 1e-5)
train['gaming_ratio'] = train['gaming_hours'] / (train['daily_screen_time_hours'] + 1e-5)
train['notifications_per_screen_hour'] = train['notifications_per_day'] / (train['daily_screen_time_hours'] + 1e-5)
train["screen_to_sleep_ratio"] = train["daily_screen_time_hours"] / (train["sleep_hours"] + 1e-5)
train["total_active_screen_hours"] = train["daily_screen_time_hours"] + train["work_study_hours"]
train["non_sleep_screen_share"] = train["daily_screen_time_hours"] / (24 - train["sleep_hours"] + 1e-5)
train['app_opens_per_screen_hour'] = train['app_opens_per_day'] / (train['daily_screen_time_hours'] + 1e-5)
train['notif_to_open_ratio'] = train['app_opens_per_day'] / (train['notifications_per_day'] + 1e-5)
train['leisure_screen_hours'] = train['social_media_hours'] + train['gaming_hours']
train['leisure_ratio'] = train['leisure_screen_hours'] / (train['daily_screen_time_hours'] + 1e-5)
train['unaccounted_screen_time'] = train['daily_screen_time_hours'] - (train['leisure_screen_hours'] + train['work_study_hours'])
train['sleep_deprived_flag'] = (train['sleep_hours'] < 6).astype(int)
train['stress_screen_interaction'] = train['stress_level'] * train['daily_screen_time_hours']


# test dataset
test['screen_time_gap'] = test['daily_screen_time_hours'] - test['sleep_hours']
test['weekend_screen_ratio'] = test['weekend_screen_time'] / (test['daily_screen_time_hours'] + 1e-5)
test['social_media_ratio'] = test['social_media_hours'] / (test['daily_screen_time_hours'] + 1e-5)
test['gaming_ratio'] = test['gaming_hours'] / (test['daily_screen_time_hours'] + 1e-5)
test['notifications_per_screen_hour'] = test['notifications_per_day'] / (test['daily_screen_time_hours'] + 1e-5)
test["screen_to_sleep_ratio"] = test["daily_screen_time_hours"] / (test["sleep_hours"] + 1e-5)
test["total_active_screen_hours"] = test["daily_screen_time_hours"] + test["work_study_hours"]
test["non_sleep_screen_share"] = test["daily_screen_time_hours"] / (24 - test["sleep_hours"] + 1e-5)
test['app_opens_per_screen_hour'] = test['app_opens_per_day'] / (test['daily_screen_time_hours'] + 1e-5)
test['notif_to_open_ratio'] = test['app_opens_per_day'] / (test['notifications_per_day'] + 1e-5)
test['leisure_screen_hours'] = test['social_media_hours'] + test['gaming_hours']
test['leisure_ratio'] = test['leisure_screen_hours'] / (test['daily_screen_time_hours'] + 1e-5)
test['unaccounted_screen_time'] = test['daily_screen_time_hours'] - (test['leisure_screen_hours'] + test['work_study_hours'])
test['sleep_deprived_flag'] = (test['sleep_hours'] < 6).astype(int)
test['stress_screen_interaction'] = test['stress_level'] * test['daily_screen_time_hours']

In [17]:
train_FE = train.copy()
train_FE.to_csv("train_FE.csv", index=False)

In [18]:
train = pd.read_csv("train_FE.csv")

In [19]:
# Pisahkan fitur dan target
X = train.drop(columns=["id", "addicted_label"])
y = train["addicted_label"]

In [20]:
# Kolom kategorikal dan numerik
categorical_features = ["gender", "academic_work_impact"]

numeric_features = [
    col for col in X.columns
    if col not in categorical_features
]

# Preprocessing yang sama untuk seluruh model
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), numeric_features),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            categorical_features
        )
    ]
)

In [21]:
with open("best_params.json", "r") as file:
    best_params = json.load(file)

print("Parameter terbaik dari tuning:")
print(best_params)

# Hilangkan awalan 'model__' agar cocok untuk HistGradientBoostingClassifier
hgb_params = {
    key.replace("model__", ""): value
    for key, value in best_params.items()
}

# Buat pipeline final dengan parameter terbaik
final_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", HistGradientBoostingClassifier(
        **hgb_params,
        random_state=42,
        early_stopping=True
    ))
])

print(final_pipeline)

Parameter terbaik dari tuning:
{'model__l2_regularization': 5.5517216852447255, 'model__learning_rate': 0.13996426971690923, 'model__max_iter': 441, 'model__max_leaf_nodes': 23, 'model__min_samples_leaf': 35}
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numeric', StandardScaler(),
                                                  ['age',
                                                   'daily_screen_time_hours',
                                                   'social_media_hours',
                                                   'gaming_hours',
                                                   'work_study_hours',
                                                   'sleep_hours',
                                                   'notifications_per_day',
                                                   'app_opens_per_day',
                                                   'weekend_screen_time',
                                            

In [22]:
# Train pada 100% data training
final_pipeline.fit(X, y)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](27,)","['age','daily_screen_time_hours','social_media_hours',..., 'unaccounted_screen_time','sleep_deprived_flag', 'stress_screen_interaction']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,27
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dro

In [23]:
test_id = test['id']
Xfinaltest = test.drop(columns=['id'])

In [24]:
# Probabilitas seseorang masuk kelas addicted_label = 1
y_prob = final_pipeline.predict_proba(Xfinaltest)[:, 1]

# Label final berdasarkan threshold default 0.5
y_pred = final_pipeline.predict(Xfinaltest)

In [25]:
submission = pd.DataFrame({
    "id": test_id,
    "addicted_label": y_prob
})

submission.to_csv("submission3.csv", index=False)

In [26]:
# Simpan model final
joblib.dump(final_pipeline, "histgradient3_tunedFE.joblib")

print("Model final berhasil dilatih dan disimpan.")

Model final berhasil dilatih dan disimpan.


In [27]:
test.to_csv("test_FE.csv")